In [5]:
import os
import re
import pandas as pd

## 省级数据处理

In [16]:
# 来源依据：国家统计局常用“四大区域划分”
region_map = {
    # 东部
    "北京": "东部", "天津": "东部", "河北": "东部", "上海": "东部",
    "江苏": "东部", "浙江": "东部", "福建": "东部",
    "山东": "东部", "广东": "东部", "海南": "东部",

    # 中部
    "山西": "中部", "安徽": "中部", "江西": "中部",
    "河南": "中部", "湖北": "中部", "湖南": "中部",

    # 西部
    "内蒙": "西部", "广西": "西部", "重庆": "西部",
    "四川": "西部", "贵州": "西部", "云南": "西部",
    "西藏": "西部", "陕西": "西部", "甘肃": "西部",
    "青海": "西部", "宁夏": "西部", "新疆": "西部",

    # 东北
    "辽宁": "东北", "吉林": "东北", "黑龙江": "东北"
}


| 季度   | 转成时间      |
| ---- | --------- |
| 第一季度 | 3 月 31 日  |
| 第二季度 | 6 月 30 日  |
| 第三季度 | 9 月 30 日  |
| 第四季度 | 12 月 31 日 |


In [17]:
## 省级data导入

gov_data_folder = "C:/Users/86159/Desktop/本科毕业论文/data/china_data/govern"
# nation_data = "C:/Users/86159/Desktop/本科毕业论文/data/china_data/national_data.csv"

all_df = [] 

def read_real_csv(path):
    # 1. 找到表头所在行（包含“时间,”）
    with open(path, encoding="gbk", errors="ignore") as f:
        for i, line in enumerate(f):
            if line.startswith("时间,"):
                header_row = i
                break
        else:
            raise ValueError(f"{path} 中未找到表头行")

    # 2. 读取数据
    df = pd.read_csv(path, skiprows=header_row, encoding="gbk")
    
     # 3. 严格筛选合法时间行
    df = df[
        df["时间"].str.match(
            r"^\d{4}年(第一|第二|第三|第四)季度$",
            na=False
        )
    ]


    # 4. 重置索引
    df = df.reset_index(drop=True)

    return df

for file in os.listdir(gov_data_folder):
    if file.endswith('.csv'):
        province = file.replace('.csv','')# 省名直接取文件名

        file_path = os.path.join(gov_data_folder, file)
        df = read_real_csv(file_path)

        df['province'] = province
        df['region'] = region_map.get(province,"未知")

        all_df.append(df)


final_df = pd.concat(all_df, ignore_index=True)


In [18]:
# import pandas as pd
# import re

def quarter_to_datetime(series):
    """
    把 '2025年第三季度' → Timestamp('2025-09-30')
    """
    def convert(x):
        match = re.match(r"(\d{4})年(第一|第二|第三|第四)季度", x)
        if not match:
            return pd.NaT

        year = int(match.group(1))
        quarter = match.group(2)

        quarter_end_month = {
            "第一": "03-31",
            "第二": "06-30",
            "第三": "09-30",
            "第四": "12-31" # 把这里的每一季度换成对应季度的最后一天
        }

        return pd.to_datetime(f"{year}-{quarter_end_month[quarter]}")

    return series.apply(convert)

final_df['时间'] = quarter_to_datetime(final_df['时间'])


In [19]:
final_df

,时间,地区生产总值累计值(亿元),地区生产总值指数(上年同期=100)累计值(%),居民人均消费支出累计值(元),城镇居民人均消费支出累计值(元),农村居民人均消费支出累计值(元),居民人均可支配收入累计值(元),城镇居民人均可支配收入累计值(元),农村居民人均可支配收入累计值(元),建筑业企业签订合同金额累计值(亿元),province,region
0,2025-09-30,40721.0,105.5,40892.0,42686.0,24606.0,69220.0,72517.0,39293.0,33745.06,上海,东部
1,2025-06-30,26222.0,105.1,27179.0,28351.0,16535.0,46805.0,48939.0,27431.0,30820.35,上海,东部
2,2025-03-31,12735.0,105.1,14047.0,14636.0,8805.0,25766.0,26929.0,15421.0,27775.90,上海,东部
3,2024-12-31,53927.0,105.0,52722.0,54980.0,32320.0,88366.0,93095.0,45644.0,38782.03,上海,东部
4,2024-09-30,38716.0,104.7,39626.0,41396.0,23682.0,66341.0,69567.0,37266.0,34472.45,上海,东部
...,...,...,...,...,...,...,...,...,...,...,...,...
522,2022-09-30,10825.0,103.0,14769.0,17480.0,10836.0,19651.0,25531.0,11119.0,2740.79,黑龙江,东北
523,2022-06-30,6614.0,102.9,9594.0,11304.0,7114.0,12775.0,16404.0,7511.0,2171.25,黑龙江,东北
524,2022-03-31,3078.0,105.5,5146.0,5974.0,3954.0,7762.0,8766.0,6316.0,1009.55,黑龙江,东北
525,2021-12-31,15293.0,106.2,20636.0,24422.0,15225.0,27159.0,33646.0,17889.0,2983.95,黑龙江,东北


In [20]:
final_df['region'].value_counts()


region
西部    204
东部    170
中部    102
东北     51
Name: count, dtype: int64

In [15]:
mask_region = (final_df['region'] == '未知')
check = final_df[mask_region]
print(check)

           时间  地区生产总值累计值(亿元)  地区生产总值指数(上年同期=100)累计值(%)  居民人均消费支出累计值(元)  \
34 2025-09-30        18472.0                     104.5         20813.0   
35 2025-06-30        12078.0                     105.4         14044.0   
36 2025-03-31         5816.0                     105.4          7523.0   
37 2024-12-31        26315.0                     105.8         28113.0   
38 2024-09-30        18193.0                     105.8         20129.0   
39 2024-06-30        11901.0                     106.2         13511.0   
40 2024-03-31         5728.0                     105.9          7138.0   
41 2023-12-31        25020.0                     107.4         27025.0   
42 2023-09-30        17479.0                     107.3         19241.0   
43 2023-06-30        11368.0                     107.4         12667.0   
44 2023-03-31         5541.0                     105.7          6497.0   
45 2022-12-31        23796.0                     104.5         22298.0   
46 2022-09-30        16741.0          